# 2. ARIMA and SARIMA Models

**ARIMA(p,d,q)** models combine:
- **AR(p)**: autoregressive component
- **I(d)**: differencing to achieve stationarity
- **MA(q)**: moving average component

This notebook covers:
- Differencing and the ADF stationarity test
- ARIMA estimation
- Auto ARIMA for automatic order selection
- SARIMA for seasonal data
- Forecasting with confidence intervals

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
import pmdarima as pm

%matplotlib inline
np.random.seed(42)

## 2.1 Stationarity and the ADF Test

The **Augmented Dickey-Fuller (ADF)** test checks whether a series has a unit root (is non-stationary).
- $H_0$: the series has a unit root (non-stationary)
- If p-value < 0.05, we reject $H_0$ and conclude stationarity

In [ ]:
n = 300
eps = np.random.normal(0, 1, n)
rw_drift = np.cumsum(0.1 + eps)  # Random walk with drift: I(1)

print(f"ADF on level:       p-value = {adfuller(rw_drift)[1]:.6f}")
print(f"ADF on first diff:  p-value = {adfuller(np.diff(rw_drift))[1]:.6f}")

## 2.2 ARIMA Estimation

ARIMA(1,1,0) is equivalent to fitting an AR(1) model on the **differenced** series.

In [ ]:
model = ARIMA(rw_drift, order=(1, 1, 0))
results = model.fit()
print(results.summary())

## 2.3 Auto ARIMA

The `pmdarima` library automates order selection using AIC/BIC criteria. It searches over combinations of $(p,d,q)$ and returns the best model.

In [ ]:
auto_model = pm.auto_arima(rw_drift, stepwise=True, trace=True)
print(f"\nBest order: {auto_model.order}")
print(auto_model.summary())

## 2.4 SARIMA for Seasonal Data

**SARIMA** extends ARIMA with seasonal components: SARIMA$(p,d,q) \times (P,D,Q)_s$.
We simulate monthly temperatures with period $s=12$.

In [ ]:
t = np.arange(360)
seasonal = 15 + 0.01*t + 10*np.sin(2*np.pi*t/12) + np.random.normal(0, 2, 360)
idx = pd.date_range('1990-01', periods=360, freq='ME')
ts = pd.Series(seasonal, index=idx)

model_s = SARIMAX(ts, order=(1,0,1), seasonal_order=(1,1,1,12))
results_s = model_s.fit(disp=False)
print(results_s.summary())

## 2.5 Forecasting with Confidence Intervals

A good forecast comes with **uncertainty quantification**. The confidence band widens as we forecast further ahead.

In [ ]:
forecast = results_s.get_forecast(steps=24)
pred = forecast.predicted_mean
ci = forecast.conf_int()

fig, ax = plt.subplots(figsize=(12, 5))
ts[-60:].plot(ax=ax, label='Observed')
pred.plot(ax=ax, label='Forecast', color='red')
ax.fill_between(ci.index, ci.iloc[:,0], ci.iloc[:,1],
                alpha=0.2, color='red')
ax.legend()
ax.set_title('SARIMA Forecast with 95% Confidence Interval')
plt.tight_layout()
plt.show()

## Key Takeaways

| Concept | Description |
|---|---|
| ADF test | Tests for unit root (non-stationarity) |
| Differencing ($d$) | Makes a series stationary |
| ARIMA$(p,d,q)$ | AR + differencing + MA |
| SARIMA | Adds seasonal components $(P,D,Q)_s$ |
| Auto ARIMA | Automatic order selection via AIC/BIC |